In [ ]:
!git clone https://github.com/dshipley71/mcp-rag.git

In [ ]:
%cd mcp-rag

In [ ]:
!pip install -r requirements.txt

In [ ]:
!apt-get remove -y nodejs npm
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!node -v
!npm -v

In [ ]:
# !npx -y @modelcontextprotocol/server-filesystem /content/mcp-rag/docs

In [ ]:
%%writefile pytest.ini
[pytest]
asyncio_mode = auto
asyncio_default_fixture_loop_scope = function

In [ ]:
%ls
%pwd

In [ ]:
import pytest
!pytest tests/test_structure.py -v

In [ ]:
import pytest
!pytest tests/test_ingestion.py -v

In [ ]:
from pathlib import Path

REPO_ROOT = Path("/content/mcp-rag").resolve()
DOCS_DIR = REPO_ROOT / "docs"
DB_DIR = REPO_ROOT / ".rag" / "velocirag"

print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR:", DOCS_DIR)
print("DB_DIR:", DB_DIR)

In [ ]:
import shutil
import subprocess
import os

if DB_DIR.exists():
    shutil.rmtree(DB_DIR)

result = subprocess.run(
    ["velocirag", "index", str(DOCS_DIR), "--db", str(DB_DIR)],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    env={**os.environ, "VELOCIRAG_DB": str(DB_DIR)},
)

print("RETURN CODE:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

In [ ]:
import os

os.environ.setdefault("MCP_FILESYSTEM_ROOT", str(DOCS_DIR))
os.environ.setdefault("VELOCIRAG_DB", str(DB_DIR))
os.environ.setdefault("RETRIEVAL_MCP_COMMAND", "velocirag")
os.environ.setdefault("UNSTRUCTURED_MCP_COMMAND", "uns_mcp")


In [ ]:
from src.config import load_catalog
from src.mcp_runtime import MCPRuntime

catalog = load_catalog("mcp_catalog.yaml")
runtime = MCPRuntime(catalog=catalog)

await runtime.connect()

print("VelociRAG tools:", await runtime.velocirag.list_tools())
print("Filesystem tools:", await runtime.filesystem.list_tools())


In [ ]:
import rich

health = await runtime.velocirag.call_tool("health", {})
rich.print("HEALTH:", health)

In [ ]:
search = await runtime.velocirag.call_tool(
    "search",
    {"query": "VelociRAG", "limit": 3}
)
rich.print("SEARCH:", search)

In [ ]:
result = await runtime.filesystem.call_tool(
    "read_text_file",
    {"path": str(DOCS_DIR / "VelociRAG.md")}
)
rich.print("FILESYSTEM READ:", result)

In [ ]:
# Parser-based ingestion test

from src.ingestion import ingest_file

test_path = str(DOCS_DIR / "VelociRAG.md")

with open(test_path, "w") as f:
    f.write("This is a test document for parser ingestion.")

await runtime.connect_ingestion()

result = await ingest_file(test_path, runtime)

print("Ingestion result:", result)

In [ ]:
from src.retrieval import run_bm25_search, run_vector_search, fetch_documents

bm25 = await run_bm25_search(runtime, "What is VelociRAG?")
vector = await run_vector_search(runtime, "What is VelociRAG?")

print("BM25:", bm25[:2])
print("VECTOR:", vector[:2])

combined = bm25 + vector
chunks = await fetch_documents(runtime, combined[:2])

print("NUM CHUNKS:", len(chunks))
if chunks:
    print("FIRST CHUNK PREVIEW:\n")
    print(chunks[0].text[:1000])

In [ ]:
from src.orchestrator import run_query
from src.models import QueryRequest

result = await run_query(
    QueryRequest(query="What is VelociRAG?"),
    runtime
)
print(result)

In [ ]:
# Optional: only run at the very end of the notebook session
try:
    await runtime.close()
except BaseException:
    pass